# 17 — Docker for AI Engineers

**Fase:** 4 — Sky & DevOps | **Tid:** 2 timer | **Krav:** Notatbok 05, 08

**Hva du bygger:** En containerisert RAG-API med Dockerfile, docker-compose for lokal stack, og GitHub Actions CI/CD-pipeline.

---

## Hvorfor Docker?

```
Uten Docker:  "Det fungerer på min maskin" 🤷
Med Docker:   Samme container kjører lokalt, i CI, og i sky ✅
```

Som AI Engineer trenger du Docker fordi:
- AI-apper har mange avhengigheter (ulike bibliotekversjoner)
- Vektordatabaser (Qdrant, Weaviate) distribueres som Docker-images
- Cloud-tjenester (Azure Container Apps, AWS ECS) kjører containers
- CI/CD-pipelines bygger og tester Docker-images

In [ ]:
import pathlib

app_dir = pathlib.Path("rag_api")
app_dir.mkdir(exist_ok=True)

(app_dir / "main.py").write_text("""\
from fastapi import FastAPI
from pydantic import BaseModel
import chromadb
from sentence_transformers import SentenceTransformer

app = FastAPI(title="Pensjons-RAG API")
embed = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
db    = chromadb.PersistentClient(path="/data/chroma")
kol   = db.get_or_create_collection("docs", metadata={"hnsw:space": "cosine"})

class Spørsmål(BaseModel):
    tekst: str
    topp_k: int = 3

@app.get("/helse")
def helse():
    return {"status": "ok", "dokumenter": kol.count()}

@app.post("/søk")
def søk(q: Spørsmål):
    sv = embed.encode([q.tekst]).tolist()
    res = kol.query(query_embeddings=sv, n_results=q.topp_k)
    return {"treff": res["documents"][0]}
""")

(app_dir / "requirements.txt").write_text("""\
fastapi==0.111.0
uvicorn==0.29.0
chromadb==0.5.0
sentence-transformers==3.0.0
""")
print("App-filer skrevet.")

---

## Del 1: Dockerfile

In [ ]:
(app_dir / "Dockerfile").write_text("""\
# Steg 1: Builder — installer avhengigheter
FROM python:3.11-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --no-cache-dir --target=/deps -r requirements.txt

# Steg 2: Runtime — kun det vi trenger (gir lite image)
FROM python:3.11-slim AS runtime
WORKDIR /app
COPY --from=builder /deps /usr/local/lib/python3.11/site-packages
COPY main.py .

# Sikkerhet: kjør som ikke-root
RUN useradd -r appuser && chown -R appuser /app
USER appuser

VOLUME ["/data"]
EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=5s CMD curl -f http://localhost:8000/helse || exit 1
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
""")

print("Dockerfile:")
print((app_dir / "Dockerfile").read_text())

**Multi-stage build forklart:**

```
Steg 1 (builder): Installer pakker (store verktøy, caches)
Steg 2 (runtime): Kopier kun det ferdige resultatet

Resultat: Image ~200 MB i stedet for ~1.5 GB
```

---

## Del 2: docker-compose for lokal stack

In [ ]:
pathlib.Path("docker-compose.yml").write_text("""\
version: '3.9'

services:
  api:
    build: ./rag_api
    ports: ["8000:8000"]
    volumes:
      - chroma_data:/data
    environment:
      - OLLAMA_URL=http://ollama:11434
    depends_on: [ollama]
    restart: unless-stopped

  ollama:
    image: ollama/ollama:latest
    ports: ["11434:11434"]
    volumes:
      - ollama_data:/root/.ollama
    restart: unless-stopped

  qdrant:
    image: qdrant/qdrant:latest
    ports: ["6333:6333"]
    volumes:
      - qdrant_data:/qdrant/storage
    restart: unless-stopped

volumes:
  chroma_data:
  ollama_data:
  qdrant_data:
""")

print("docker-compose.yml skrevet.")
print("\nKjør med:  docker-compose up -d")
print("Test:      curl http://localhost:8000/helse")
print("Stopp:     docker-compose down")

---

## Del 3: GitHub Actions CI/CD

In [ ]:
gha_dir = pathlib.Path(".github/workflows")
gha_dir.mkdir(parents=True, exist_ok=True)

(gha_dir / "ci.yml").write_text("""\
name: CI

on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
          cache: 'pip'
      - run: pip install -r rag_api/requirements.txt pytest
      - run: pytest tests/ -v

  docker:
    runs-on: ubuntu-latest
    needs: test
    steps:
      - uses: actions/checkout@v4
      - name: Build image
        run: docker build -t rag-api:${{ github.sha }} ./rag_api
      - name: Smoke test
        run: |
          docker run -d --name test-api -p 8000:8000 rag-api:${{ github.sha }}
          sleep 5
          curl -f http://localhost:8000/helse
          docker stop test-api
""")
print("GitHub Actions workflow skrevet.")

---

## Nyttige Docker-kommandoer

In [ ]:
print("""
# Bygg image
docker build -t rag-api:latest ./rag_api

# Kjør med vedvarende data
docker run -d -p 8000:8000 -v $(pwd)/data:/data rag-api:latest

# Se logger
docker logs -f <container_id>

# Debug inne i container
docker exec -it <container_id> bash

# Sjekk image-størrelse
docker image ls

# Start/stopp hele stacken
docker-compose up -d
docker-compose down
""")

---

## Oppsummering

| Konsept | Hva det er |
|---------|----------|
| Dockerfile | Oppskrift for å bygge et image |
| Multi-stage build | Skille bygging fra kjøring → lite image |
| VOLUME | Data som overlever container-restart |
| docker-compose | Koordiner flere containers lokalt |
| GitHub Actions | Automatisk bygging og testing på push |

---

## Hva er neste steg?

**Neste: `18_azure_ai_foundry.ipynb`** — Distribuer til sky: Azure AI Foundry, Container Apps, Key Vault og managed identity.